# doc-extraction — OmniDocBench full benchmark (Kaggle, T4 GPU)

Thin orchestrator notebook — 10 steps, no extraction or evaluation logic
implemented here. Everything runs through the repo's own scripts
(`experiments/005_omnidocbench/run.py`,
`src/doc_extraction/evaluation/omnidocbench.py`). See
[`experiments/005_omnidocbench/README.md`](../README.md) for the full design,
the pinned OmniDocBench commit, and what has/hasn't been run locally, and
[`docs/kaggle.md`](../../../docs/kaggle.md) for the step-by-step setup this
notebook assumes.

**No private data.** Only the public OmniDocBench dataset is used here. This
repo's own `data/` (local/private sample documents) is gitignored and is not
part of this clone — never attach it as a Kaggle input.

**CPU pipeline vs. GPU backend vs. benchmark evaluator** — three different
things that are easy to conflate:
- The **pipeline** (`baseline`, `docling` backends) runs on **CPU** by
  default (`configs/cpu.yaml`, `device: cpu`) whether or not a GPU is
  attached — neither backend currently requests CUDA.
- The **T4 GPU** this notebook checks for (Step 2) is only actually used
  once a GPU-requesting backend exists (see `docs/backends.md`); today it
  buys faster page rendering/model inference *if* a backend uses it, not
  automatically.
- The **OmniDocBench evaluator** (Step 4/8) is a separate, CPU-only, Python
  \<3.12 subprocess that scores predictions — it never touches the GPU.

## Step 1 — Check Python

In [ ]:
import sys
print(sys.version)
assert sys.version_info >= (3, 10), "doc_extraction needs Python 3.10+"

## Step 2 — Check GPU

Diagnostic only (see the CPU/GPU/evaluator note above) — nothing later in
this notebook currently changes behavior based on this.

In [ ]:
import torch

has_cuda = torch.cuda.is_available()
print(f"CUDA available: {has_cuda}")
if has_cuda:
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU visible — check Settings > Accelerator is set to GPU T4 x2 (or T4 x1).")

## Step 3 — Clone repo

Idempotent — reruns of this cell (e.g. after a kernel restart mid-session)
reuse the existing clone instead of failing on `git clone` into a
non-empty directory, and pull the latest commit so a stale clone from an
earlier session in the same Kaggle workspace doesn't shadow new fixes.

In [ ]:
import os

REPO_DIR = "/kaggle/working/doc-extraction"

if not os.path.exists(REPO_DIR):
    os.chdir("/kaggle/working")
    !git clone https://github.com/<YOUR_USERNAME>/doc-extraction.git
else:
    print("Repository already exists:", REPO_DIR, "- pulling latest")
    !git -C {REPO_DIR} pull --ff-only

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

## Step 4 — Install deps

Two independent installs: this project's own package (on Kaggle's default
Python), and the OmniDocBench evaluator (a separate project, pinned to the
exact commit this repo's adapter was verified against — see the parent
README's "Upstream, pinned exactly" table).

The evaluator needs Python `>=3.10,<3.12`. Kaggle's default Python image is
**3.12+**, which the evaluator's own `pyproject.toml` rejects — so this step
always provisions an isolated Python 3.11 venv for it at `.venv-omnidoc`,
the same layout `experiments/005_omnidocbench/README.md` describes for the
local dev machine. [`uv`](https://docs.astral.sh/uv/) makes fetching a
specific Python version and building that venv fast and reliable inside a
Kaggle session; it is only used for this one install, not for the main
project.

In [ ]:
# This project. Kaggle's default Python satisfies its own constraints
# directly — no isolated venv needed here.
!pip install -e ".[docling,tables]" -q

import doc_extraction
print("doc_extraction imported OK:", doc_extraction.__file__)

### Prefetch Docling models

**Not optional here**, despite how `docs/setup.md` phrases it for the
local dev machine: once `DOCLING_ARTIFACTS_PATH` is set to anything (which
the lines below do — redirecting the ~1.5 GB of models onto the Kaggle
working volume instead of the small root volume), Docling refuses to
auto-download and requires that path to already contain the models,
raising `RuntimeError: ... is not valid` otherwise. Prefetch once, up
front, rather than discovering that failure mid-run.

In [ ]:
import os

CACHE_ROOT = "/kaggle/working/.cache"
os.makedirs(f"{CACHE_ROOT}/huggingface", exist_ok=True)
os.makedirs(f"{CACHE_ROOT}/docling", exist_ok=True)
os.environ["HF_HOME"] = f"{CACHE_ROOT}/huggingface"
os.environ["DOCLING_ARTIFACTS_PATH"] = f"{CACHE_ROOT}/docling"
os.environ["XDG_CACHE_HOME"] = CACHE_ROOT

# Default model set (layout, table structure, ...) plus EasyOCR for the
# en/vi languages this project's configs/cpu.yaml is configured for.
!python -m docling.cli.tools models download -o {CACHE_ROOT}/docling
!python -m docling.cli.tools models download easyocr \
    --easyocr-lang en --easyocr-lang vi -o {CACHE_ROOT}/docling

In [ ]:
import os

# OmniDocBench evaluator source: cloned outside the repo (its own
# Apache-2.0 project, not vendored) and pinned to the commit the adapter
# was verified against.
OMNIDOC_REPO = f"{REPO_DIR}/.external/OmniDocBench"

if not os.path.exists(OMNIDOC_REPO):
    !mkdir -p .external
    !git clone https://github.com/opendatalab/OmniDocBench.git {OMNIDOC_REPO}
!git -C {OMNIDOC_REPO} fetch --all --tags -q
!git -C {OMNIDOC_REPO} checkout 193627ae9e97d89188468ed1ee3b7a856ff76044

In [ ]:
# Isolated Python 3.11 venv for the evaluator, via uv.

!pip install -q -U uv
!uv python install 3.11

OMNIDOC_VENV = f"{REPO_DIR}/.venv-omnidoc"
OMNIDOC_PYTHON = f"{OMNIDOC_VENV}/bin/python"

import os
if not os.path.exists(OMNIDOC_VENV):
    !uv venv --python 3.11 {OMNIDOC_VENV}

!uv pip install --python {OMNIDOC_PYTHON} -U pip setuptools wheel -q
!uv pip install --python {OMNIDOC_PYTHON} -e {OMNIDOC_REPO} -q

# Verify — the installed distribution is named omnidocbench-eval but its
# importable top-level module is `src` (a src-layout package); see
# experiments/005_omnidocbench/README.md "Setup" for this exact check.
!{OMNIDOC_PYTHON} -c "from src.core.pipeline import run_config_file; print('evaluator import OK')"

## Step 5 — Locate OmniDocBench dataset

Option A (recommended, for the real benchmark): attach the dataset as a
Kaggle Dataset input — it appears under `/kaggle/input/<dataset-name>/`.
Edit `KAGGLE_DATASET_NAME` below to match what you attached via "Add
Input". Option B (automatic fallback, used if nothing is attached): the
small 18-page official demo set bundled with the evaluator clone in
Step 4 — enough to prove the pipeline end-to-end, not the full benchmark.
Either way, only public OmniDocBench data — never this repo's own `data/`.

In [ ]:
import os

# Option A: edit this to the Kaggle Dataset you attached via "Add Input".
# Leave as-is (or anything that doesn't exist) to fall back to Option B.
KAGGLE_DATASET_NAME = "<dataset-name>"
_attached_path = f"/kaggle/input/{KAGGLE_DATASET_NAME}"
_demo_path = f"{REPO_DIR}/.external/OmniDocBench/demo_data/omnidocbench_demo"

if os.path.exists(_attached_path):
    DATASET_PATH = _attached_path
else:
    print(f"No Kaggle Dataset attached at {_attached_path} — falling back to "
          f"the bundled 18-page demo set. Attach a dataset via 'Add Input' and "
          f"set KAGGLE_DATASET_NAME above to run the real benchmark instead.")
    DATASET_PATH = _demo_path

OUTPUT_ROOT = "/kaggle/working/results"

## Step 6 — Print dataset path

In [ ]:
assert os.path.exists(DATASET_PATH), (
    f"DATASET_PATH does not exist: {DATASET_PATH} — if this is the demo-set "
    f"fallback, re-run Step 4 first (it clones .external/OmniDocBench)."
)
print(f"DATASET_PATH = {DATASET_PATH}")
print(sorted(os.listdir(DATASET_PATH))[:20])

## Step 7 — Run small smoke test

Validate the integration on a small, deterministic subset before spending
GPU/CPU time on the full run — same principle as the local CPU validation
(see the main README's "Local validation").

In [ ]:
!python experiments/005_omnidocbench/run.py \
    --dataset {DATASET_PATH} \
    --backend baseline \
    --output {OUTPUT_ROOT}/baseline_smoke \
    --subset 20 \
    --match-workers 2

## Step 8 — Run full experiment

Omit `--subset` for the full dataset. `--match-workers`: keep to roughly
1/3-1/2 of the instance's CPU count (upstream's own guidance, to avoid
deadlocks/OOM in its worker pools) — the evaluator itself is CPU-bound
regardless of the attached accelerator.

**If Step 5 fell back to the bundled demo set**, this still only processes
those same 18 pages — attach a real Kaggle Dataset (Step 5, Option A) to
run the actual 1651-page benchmark.

`docling` is included as a second arm for comparison; both currently run
on CPU (see the CPU/GPU/evaluator note at the top). Do not claim a backend
used the T4 unless Step 2 showed CUDA available *and* the backend actually
requests `device: cuda` (see `configs/gpu.yaml` — documented, unvalidated).

In [ ]:
!python experiments/005_omnidocbench/run.py \
    --dataset {DATASET_PATH} \
    --backend baseline \
    --output {OUTPUT_ROOT}/baseline \
    --match-workers 4

In [ ]:
!python experiments/005_omnidocbench/run.py \
    --dataset {DATASET_PATH} \
    --backend docling \
    --output {OUTPUT_ROOT}/docling \
    --match-workers 4

## Step 9 — Print metrics

In [ ]:
from pathlib import Path

for backend_dir in sorted(Path(OUTPUT_ROOT).glob("*")):
    report = backend_dir / "report.md"
    if report.exists():
        print(f"===== {backend_dir.name} =====")
        print(report.read_text(encoding="utf-8"))
        print()

## Step 10 — Save results

`/kaggle/working` persists as the notebook's output and can be downloaded
from the output panel after the session ends. To fold results back into
the repo's own history, copy just the small committed-shape files
(`report.md`, `metrics.json`, `runtime.json`, `run_metadata.json` — not
`predictions/` or the evaluator's raw debug dumps) into
`experiments/005_omnidocbench/results/<backend>/`, matching the local-run
convention described in the parent README's "Files" section.

In [ ]:
import shutil

for backend in ("baseline", "docling"):
    src = Path(OUTPUT_ROOT) / backend
    if not src.exists():
        continue
    for name in ("report.md", "metrics.json", "runtime.json", "run_metadata.json"):
        f = src / name
        if f.exists():
            print(f"kept for output panel: {f}")
# Files above remain under /kaggle/working; download them from the output
# panel, then copy locally into experiments/005_omnidocbench/results/.